In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/breast-cancer-detection/sample-submission.csv
/kaggle/input/competitions/breast-cancer-detection/train.csv
/kaggle/input/competitions/breast-cancer-detection/test.csv


In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
from sklearn.model_selection import GridSearchCV

warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('/kaggle/input/competitions/breast-cancer-detection/train.csv')
test = pd.read_csv('/kaggle/input/competitions/breast-cancer-detection/test.csv')

print("Dataset Info:")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTarget distribution:")
print(train['diagnosis'].value_counts())

# Prepare data
X = train.drop(['id', 'diagnosis'], axis=1)
y = train['diagnosis']
X_test = test.drop(['id'], axis=1)

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)  # M=1, B=0

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Use StratifiedKFold for more robust validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\n{'='*50}")
print("Performing GridSearchCV with StratifiedKFold...")
print('='*50)

# Optimized parameter grid based on your best results
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.1, 0.2, 0.3],
    'n_estimators': [150, 200, 250],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

# Reduce grid for faster execution (optional)
# param_grid = {
#     'max_depth': [3, 4],
#     'learning_rate': [0.2, 0.3],
#     'n_estimators': [200],
#     'subsample': [0.9, 1.0],
#     'colsample_bytree': [0.9, 1.0],
#     'min_child_weight': [3],
#     'gamma': [0.1]
# }

grid_search = GridSearchCV(
    xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False,
        verbosity=0
    ),
    param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_scaled, y)

print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# Get best parameters
best_params = grid_search.best_params_.copy()

# Train final model with all data and use all training data for final model
print("\n" + "="*50)
print("Training final model on ALL training data...")
print("="*50)

# Use the best parameters found
final_model = xgb.XGBClassifier(
    **best_params,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
    verbosity=0
)

# Train on ALL training data (no validation split) since we already have CV scores
final_model.fit(X_scaled, y)

# ============================================
# Evaluate using cross-validation scores we already have
# ============================================
print(f"\n{'='*50}")
print("Model Performance Summary:")
print(f"{'='*50}")
print(f"Best Hyperparameters: {best_params}")
print(f"GridSearch CV Score (5-fold): {grid_search.best_score_:.4f}")

# Get individual fold scores for more insight
cv_results = grid_search.cv_results_
mean_scores = cv_results['mean_test_score']
std_scores = cv_results['std_test_score']
print(f"\nTop 3 parameter combinations:")
for idx in np.argsort(mean_scores)[-3:][::-1]:
    print(f"  Score: {mean_scores[idx]:.4f} (+/- {std_scores[idx]:.4f})")
    print(f"  Params: {cv_results['params'][idx]}")

# ============================================
# Feature importance with best model
# ============================================
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

print("\nTop 15 Most Important Features:")
print(feature_importance.to_string(index=False))

# ============================================
# Predict and Create Submission
# ============================================
test_pred = final_model.predict(X_test_scaled)
test_pred_labels = le.inverse_transform(test_pred)

submission = pd.DataFrame({
    'Id': test['id'],
    'diagnosis': test_pred_labels
})
submission.to_csv('submission.csv', index=False)

print(f"\n{'='*50}")
print("✓ Submission saved: submission.csv")
print(f"{'='*50}")
print("\nSubmission preview:")
print(submission.head(10))
print(f"\nPredictions distribution:")
print(submission['diagnosis'].value_counts())

# Optional: Add prediction probabilities for confidence
test_proba = final_model.predict_proba(X_test_scaled)
print(f"\nPrediction confidence (sample):")
for i in range(5):
    prob_benign = test_proba[i][0]
    prob_malignant = test_proba[i][1]
    pred_class = 'B' if test_pred[i] == 0 else 'M'
    print(f"  Sample {i+1}: B={prob_benign:.3f}, M={prob_malignant:.3f} -> Predicted: {pred_class}")

Dataset Info:
Train shape: (398, 33)
Test shape: (171, 32)

Target distribution:
diagnosis
B    242
M    156
Name: count, dtype: int64

Performing GridSearchCV with StratifiedKFold...
Fitting 5 folds for each of 2187 candidates, totalling 10935 fits

Best parameters found: {'colsample_bytree': 0.9, 'gamma': 0.2, 'learning_rate': 0.3, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 150, 'subsample': 0.9}
Best CV score: 0.9698

Training final model on ALL training data...

Model Performance Summary:
Best Hyperparameters: {'colsample_bytree': 0.9, 'gamma': 0.2, 'learning_rate': 0.3, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 150, 'subsample': 0.9}
GridSearch CV Score (5-fold): 0.9698

Top 3 parameter combinations:
  Score: 0.9698 (+/- 0.0282)
  Params: {'colsample_bytree': 0.9, 'gamma': 0.2, 'learning_rate': 0.3, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 150, 'subsample': 0.9}
  Score: 0.9698 (+/- 0.0282)
  Params: {'colsample_bytree': 0.9, 'gamma': 0.2, 'l